# **Part A — Standup and budget**

**Done :-**
* Defined the Bank Marketing capstone scope and target variable `y`.
* Built an initial classification baseline and model.
* Measured split-to-split score variation on Day 42.
* Tested six candidate features on Day 43.
* Found that individual feature gains were very small compared with the observed noise spread.

**Blocked :-**
* Need to verify that the existing train/test split and preprocessing are consistent.
* The best model configuration and its generalisation performance still need to be evaluated.
* The model's worst errors and weakest customer segments have not yet been fully analysed.

**Today :-**
* Tune the model using RandomizedSearchCV and cross-validation.
* Compare tuned and untuned performance using Average Precision (AP).
* Compare the tuning gain against the Day 42 noise spread of 0.042.
* Inspect the 20 worst cases and analyse error rates across meaningful feature groups.
* Select and document the final model.

**Budget before tuning**  
**Tuning Budget -- Set Before Search**  
* Search Method - RandomizedsearchCV
* Maximum Configuration - 20
* Cross-Validation Folds - 3
* Maximum planned model fits - 60
* Random Seeds - 42
* Evaluation Metric - Average Precision (AP)
* Stopping Rule - Stop after the planned 20 configurations. Do not extend the search just because the score is disappointing.
* Expected Tuning Gain - +0.005 AP over the untuned model.
* Day42 Reference noise spread - 0.042 AP, subject to verifying that both measurements use the same metric and evaluation setup.

**Why this budget?**
* Twenty configurations with three-fold cross-validation give a defined search budget. The expected gain of +0.005 AP is a hypothesis, not a guaranteed result. The actual gain will be compared against the measured noise after tuning.

# **Part B — Tune properly**

In [1]:
# Load data and build models
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from scipy.stats import randint, uniform
from sklearn.base import clone

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, recall_score, precision_score, confusion_matrix
from IPython.display import display

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
data_path = Path("../data/bank-full.csv")

df = pd.read_csv(data_path, sep=";")

print("Dataset Loaded Successfully")
print("Rows:", df.shape[0])
print("Columns", df.shape[1])

display(df.head())

Dataset Loaded Successfully
Rows: 45211
Columns 17


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
# inspect the target and prepare features
print("Target Distribution:")
print(df["y"].value_counts())

print("\nTarget Proportions:")
print(df["y"].value_counts(normalize=True).round(4))

Target Distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target Proportions:
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


In [4]:
# Exclude target and post-call duration
X = df.drop(columns=["y", "duration"].copy())
y = df["y"].map({"no": 0, "yes": 1})

if y.isna().any():
    raise ValueError("Unexpected target values found in column y.")

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)
print("Positive Rate:", round(y.mean(), 4))

Feature Shape: (45211, 15)
Target Shape: (45211,)
Positive Rate: 0.117


In [5]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training Rows:", X_train.shape[0])
print("Testing Rows:", X_test.shape[0])
print("Training Positive Rate:", round(y_train.mean(), 4))
print("Testing Positive Rate:", round(y_test.mean(), 4))

Training Rows: 36168
Testing Rows: 9043
Training Positive Rate: 0.117
Testing Positive Rate: 0.117


In [6]:
# Define Preprocessing 
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

Numeric Features: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
Categorical Features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [7]:
# Strong baseline: Logistic Regression
strong_features = ["age", "job", "marital", "education", "housing", "loan"]
strong_numeric = ["age"]
strong_categorical = ["job", "marital", "education", "housing", "loan"]

strong_preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, strong_numeric),
    ("categorical", categorical_preprocessor, strong_categorical)
])
strong_baseline = Pipeline([
    ("Preprocessing", strong_preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])
strong_baseline.fit(X_train[strong_features], y_train)

strong_scores = strong_baseline.predict_proba(X_test[strong_features])[:, 1]
strong_ap = average_precision_score(y_test, strong_scores)

print("Strong Logistic Regression AP:", round(strong_ap, 4))

Strong Logistic Regression AP: 0.2142


In [8]:
# Untuned Random Forest
first_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])
first_model.fit(X_train, y_train)

model_scores = first_model.predict_proba(X_test)[:, 1]
model_ap = average_precision_score(y_test, model_scores)

print("Untuned Random Forest AP:", round(model_ap, 4))
print("Strong Logistic Regression AP:", round(strong_ap, 4))

Untuned Random Forest AP: 0.4284
Strong Logistic Regression AP: 0.2142


In [9]:
# Tune Random Forest using RandomizedSearchCV
param_distribution = {
    "model__n_estimators": randint(100, 401),
    "model__max_depth": [None, 10, 15, 20, 30],
    "model__min_samples_split": randint(2, 21),
    "model__min_samples_leaf": randint(1, 11),
    "model__max_features": ["sqrt", "log2", 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

random_search = RandomizedSearchCV(estimator=first_model, param_distributions=param_distribution, n_iter=20,
                                  scoring="average_precision", cv=3, random_state=42, return_train_score=True,
                                  refit=True, verbose=1)

random_search.fit(X_train, y_train)

tuned_model = random_search.best_estimator_


print("Best Cross-Validation AP:", round(random_search.best_score_, 4))

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nConfigurations Evaluated:", len(random_search.cv_results_["params"]))

print("Planned maximum candidate-fold fits:", 20 * 3)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Cross-Validation AP: 0.443

Best Parameters:
{'model__class_weight': None, 'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 11, 'model__n_estimators': 287}

Configurations Evaluated: 20
Planned maximum candidate-fold fits: 60


In [10]:
# Review all tuning results
cv_results = pd.DataFrame(random_search.cv_results_)

results_table = cv_results[[
    "rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"
]].sort_values("rank_test_score")

display(results_table.round(4))

,rank_test_score,mean_test_score,std_test_score,mean_train_score,params
4,1,0.4430,0.0073,0.7930,"{'model__class_weight': None, 'model__max_dept..."
14,2,0.4413,0.0087,0.7038,"{'model__class_weight': None, 'model__max_dept..."
12,3,0.4404,0.0089,0.7513,"{'model__class_weight': None, 'model__max_dept..."
1,4,0.4395,0.0102,0.6683,"{'model__class_weight': 'balanced_subsample', ..."
16,5,0.4394,0.0085,0.8443,"{'model__class_weight': 'balanced_subsample', ..."
8,6,0.4391,0.0098,0.7270,"{'model__class_weight': 'balanced_subsample', ..."
17,7,0.4378,0.0103,0.7301,"{'model__class_weight': 'balanced', 'model__ma..."
5,8,0.4371,0.0107,0.7561,"{'model__class_weight': 'balanced_subsample', ..."
18,9,0.4362,0.0088,0.7869,"{'model__class_weight': 'balanced', 'model__ma..."
0,10,0.4361,0.0083,0.6295,"{'model__class_weight': 'balanced_subsample', ..."


In [11]:
best_cv_mean = random_search.best_score_

best_index = random_search.best_index_
best_cv_std = cv_results.loc[best_index, "std_test_score"]

print("Best mean cv AP:", round(best_cv_mean, 4))
print("Best candidate CV standard deviation:", round(best_cv_std, 4))

Best mean cv AP: 0.443
Best candidate CV standard deviation: 0.0073


# **Part C — Did tuning actually help?**

In [12]:
# Compare all three models on the same holdout
tuned_scores = tuned_model.predict_proba(X_test)[:, 1]
tuned_ap = average_precision_score(y_test, tuned_scores)

comparison = pd.DataFrame([
    {
        "Model": "Strong Logistic Regression", 
        "Holdout AP": strong_ap
    }, {
        "Model": "Untuned Random Forest",
        "Holdout AP": model_ap
    }, {
        "Model": "Tuned Random Forest",
        "Holdout AP": tuned_ap
    }
])


display(comparison.round(4))

,Model,Holdout AP
0,Strong Logistic Regression,0.2142
1,Untuned Random Forest,0.4284
2,Tuned Random Forest,0.4580


In [13]:
tuning_gain = tuned_ap - model_ap
print("Tuning gain over untuned Random Forest:", round(tuning_gain, 4))
print("Precommitted target (+0.005) achieved:", tuning_gain >= 0.005)

Tuning gain over untuned Random Forest: 0.0296
Precommitted target (+0.005) achieved: True


In [14]:
# Measure split-to-split variation
split_seeds = [42, 100, 2026]
split_noise_results = []

for seed in split_seeds:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, random_state=seed, test_size=0.20, stratify=y)
    
    model_for_split = clone(first_model)
    model_for_split.fit(X_tr, y_tr)

    val_scores = model_for_split.predict_proba(X_val)[:, 1]
    val_ap = average_precision_score(y_val, val_scores)

    split_noise_results.append({
        "Random State": seed,
        "Untuned Random Forest AP": val_ap
    })


split_noise_df = pd.DataFrame(split_noise_results)
display(split_noise_df.round(4))

,Random State,Untuned Random Forest AP
0,42,0.4284
1,100,0.4218
2,2026,0.4442


In [15]:
split_min = split_noise_df["Untuned Random Forest AP"].min()
split_max = split_noise_df["Untuned Random Forest AP"].max()
split_spread = split_max - split_min

print("Minimum AP:", split_min.round(4))
print("Maximum AP:", split_max.round(4))
print("Split-to-split AP Spread:", split_spread.round(4))
print("Tuning Gain:", round(tuning_gain, 4))
print("Is tuning gain larger than this observed spread?", tuning_gain > split_spread)

Minimum AP: 0.4218
Maximum AP: 0.4442
Split-to-split AP Spread: 0.0225
Tuning Gain: 0.0296
Is tuning gain larger than this observed spread? True


In [16]:
# Interpretation
if tuning_gain >= 0.005 and tuning_gain > split_spread:
    print("""Interpretation: Tuning exceed the precommitted AP target
            and the observed split spread. This is encouraging evidence of improvement,
            but it is not proof of universal generalization.
    """)
elif tuning_gain > 0:
    print("""Interpretation: Tuning improved holdout AP, but the gain did not clearly
    exceed both the target and observed split spread.
    """)
else:
    print("""Interpretation: Tuning did not improve holdout AP over the untuned Random Forest.
    Prefer the simpler or better-performing model unless other evidence supports the tuned model.
    """)

Interpretation: Tuning exceed the precommitted AP target
            and the observed split spread. This is encouraging evidence of improvement,
            but it is not proof of universal generalization.
    


**Tuning Results**  
* Untuned Random Forest holdout AP: `0.4284`
* Tuned Random Forest holdout AP: `0.4580`
* Tuning gain: `0.0296`
* Best cross-validation AP: `0.443`
* Cross-validation standard deviation: `0.0073`
* Three-split AP spread: `0.0225`

**Interpretation**
* The tuned model's improvement was compared with both the precommitted target and observed split-to-split variation. The final decision will also consider the Logistic Regression baseline, error analysis, and the operational use case.

# **Part D — Error analysis**

In [17]:
# Create prediction and error table
error_analysis = X_test.copy()

error_analysis["actual"] = y_test.to_numpy()
error_analysis["predicted_probability"] = tuned_scores
error_analysis["predicted"] = (error_analysis["predicted_probability"] >= 0.5).astype(int)
error_analysis["correct"] = (error_analysis["actual"] == error_analysis["predicted"])

# confidence in the prediction, even when it is wrong
error_analysis["predicted_confidence"] = np.where(error_analysis["predicted"] == 1,
                                                 error_analysis["predicted_probability"],
                                                 1 - error_analysis["predicted_probability"])

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 1), 

        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 0)
    ], [
        "False Positive", "False Negative"
    ], default="Correct"
)


print("Total Holdout Rows:", len(error_analysis))
print("Correct Predictions:", int(error_analysis["correct"].sum()))
print("Incorrect Predictions:", int((~error_analysis["correct"]).sum()))

print("\nError Types:")
print(error_analysis["error_type"].value_counts())

Total Holdout Rows: 9043
Correct Predictions: 8103
Incorrect Predictions: 940

Error Types:
error_type
Correct           8103
False Negative     835
False Positive     105
Name: count, dtype: int64


In [18]:
# Inspect the 20 highest confidence errors
wrong_predictions = error_analysis[error_analysis["correct"] == False].copy()
worst_20_errors = wrong_predictions.sort_values("predicted_confidence", ascending=False).head(20)


columns_to_show = ["age", "job", "marital", "education", "housing", "loan",	"contact", "campaign",
                   "pdays", "previous", "poutcome", "actual", "predicted_probabiltiy", "predicted",
                  "predicted_confidence", "error_type"]

columns_to_show = [col for col in columns_to_show
                  if col in worst_20_errors.columns]

display(worst_20_errors[columns_to_show].round(4))

,age,job,marital,education,housing,loan,contact,campaign,pdays,previous,poutcome,actual,predicted,predicted_confidence,error_type
11510,55,housemaid,married,primary,no,no,unknown,2,-1,0,unknown,1,0,0.9928,False Negative
5888,34,management,married,tertiary,yes,no,unknown,4,-1,0,unknown,1,0,0.9917,False Negative
1124,29,management,married,tertiary,yes,yes,unknown,4,-1,0,unknown,1,0,0.9901,False Negative
4866,36,management,married,tertiary,yes,yes,unknown,2,-1,0,unknown,1,0,0.9893,False Negative
2377,38,management,married,tertiary,yes,no,unknown,4,-1,0,unknown,1,0,0.9893,False Negative
2650,46,blue-collar,married,primary,yes,no,unknown,2,-1,0,unknown,1,0,0.9880,False Negative
2767,52,housemaid,married,secondary,yes,no,unknown,3,-1,0,unknown,1,0,0.9865,False Negative
43874,51,management,married,tertiary,no,no,unknown,1,-1,0,unknown,1,0,0.9848,False Negative
129,55,services,married,secondary,yes,no,unknown,1,-1,0,unknown,1,0,0.9839,False Negative
5024,27,technician,married,secondary,yes,yes,unknown,1,-1,0,unknown,1,0,0.9821,False Negative


**Interpretation :-**
* `False Positive:` The model predicted subscriptions, but the actual outcome differed. This could have implications for campaign calls.
* `False Negative:` The model did not predict a subscription, but the actual outcome was 'yes'. Potential customers might be missed.
* These 20 cases are not necessarily the “hardest customers”; they are the most high-confidence incorrect predictions at a threshold of 0.5.

In [19]:
# Compare error types 
error_types_counts = (
    error_analysis["error_type"]
    .value_counts()
    .rename_axis("Error Type")
    .reset_index(name="Count")
)

display(error_types_counts)

,Error Type,Count
0,Correct,8103
1,False Negative,835
2,False Positive,105


In [20]:
# Evaluate customer segments
segment_data = X_test.copy()
segment_data["actual"] = y_test.to_numpy()
segment_data["score"] = tuned_scores

segment_results = []

for feature in ["job", "housing"]:
    for segment_value, group in segment_data.groupby(feature, dropna=False):
        if len(group) < 30:
            continue
        segment_ap = average_precision_score(group["actual"], group["score"])
        segment_prediction = (group["score"] >= 0.5).astype(int)
        segment_recall = recall_score(group["actual"], segment_prediction, zero_division=0)

        segment_results.append({
            "Feature": feature, "Segment": str(segment_value),
            "Rows": len(group), "Positive Rate": group["actual"].mean(),
            "AP": segment_ap, "Recall at 0.5": segment_recall
        })

segment_results_df = pd.DataFrame(segment_results)

display(segment_results_df.sort_values(["Feature", "AP"]).round(4))

,Feature,Segment,Rows,Positive Rate,AP,Recall at 0.5
13,housing,yes,4948,0.0766,0.3053,0.1108
12,housing,no,4095,0.1658,0.5350,0.2666
2,job,entrepreneur,301,0.0731,0.1904,0.0000
7,job,services,806,0.0881,0.3714,0.1268
1,job,blue-collar,1902,0.0720,0.3796,0.1387
8,job,student,180,0.2778,0.4113,0.1600
3,job,housemaid,227,0.1057,0.4299,0.2500
0,job,admin.,1030,0.1184,0.4472,0.2377
6,job,self-employed,336,0.1339,0.4773,0.2000
4,job,management,1947,0.1335,0.4932,0.2346


**Error Analysis Findings**
* The 20 highest-confidence errors were inspected.
* False positives can lead to unnecessary outreach.
* False negatives can cause potential subscribers to be missed.
* Segment performance was compared for job and housing groups.
* Segment metrics must be interpreted alongside group sizes and positive rates.

**Observations**
1. Most common error type: `False Negative`
2. Pattern observed in the 20 errors: `Several errors occurred among customers with housing loans.`
3. Lowest-AP segment with sufficient rows: `job = blue-collar (AP = 0.3796, rows = 1902)`
4. Possible operational implication: `False negatives may cause potential subscribers to be missed, while false positives may lead to unnecessary calls.`
5. Limitation: Error patterns are descriptive and do not establish causation

# **Part E — Choose the final model**

In [21]:
# Compare Final candidates
final_comparison = pd.DataFrame([
    {
        "Model": "Strong Logistic Regression",
        "Holdout AP": strong_ap
    }, {
        "Model": "Untuned Random Forest",
        "Holdout AP": model_ap
    }, {
        "Model": "Tuned Random Forest",
        "Holdout AP": tuned_ap
    }
]).sort_values("Holdout AP", ascending=False).reset_index(drop=True)

display(final_comparison.round(4))

,Model,Holdout AP
0,Tuned Random Forest,0.4580
1,Untuned Random Forest,0.4284
2,Strong Logistic Regression,0.2142


In [22]:
best_holdout_model = final_comparison.iloc[0]["Model"]
best_holdout_ap = final_comparison.iloc[0]["Holdout AP"]

print("Highest holdout AP:", best_holdout_model)
print("Highest holdout AP value:", round(best_holdout_ap, 4))

Highest holdout AP: Tuned Random Forest
Highest holdout AP value: 0.458


In [23]:
# Apply a transparent decision rule
if tuning_gain > 0.005 and tuning_gain > split_spread:
    final_model_name = "Tuned Random Forest"
    final_model = tuned_model
    final_model_scores = tuned_scores
    final_model_ap = tuned_ap

else:
    if strong_ap >= model_ap:
        final_model_name = "Strong Logistic Regression"
        final_model = strong_baseline
        final_model_scores = strong_scores
        final_model_ap = strong_ap

    else:
        final_model_name = "Untuned Random Forest"
        final_model = first_model
        final_model_scores = model_scores
        final_model_ap = model_ap


print("Selected Model:", final_model_name)
print("Selected model holdout AP:", round(final_model_ap, 4))
print("Tuning gain", round(tuning_gain, 4))
print("Observed split spread:", round(split_spread, 4))

Selected Model: Tuned Random Forest
Selected model holdout AP: 0.458
Tuning gain 0.0296
Observed split spread: 0.0225


**Decision Log:-**  
**Final Model Decision**  
* Selected Model :- Tuned Random Forest

**Evidence**  
* Strong Logistic Regression AP :- 0.2142
* Untuned Random Forest AP :- 0.4284
* Tuned Random Forest AP :- 0.4580
* Tuning Gain :- 0.0296
* Three-Split AP Spread :- 0.0225

**Why this model?**
* I selected **Tuned Random Forest** because it achieved a holdout Average Precision (AP) of **0.458**.
* The untuned Random Forest achieved an AP of **0.4284**, while the tuned Random Forest achieved an AP of **0.4580**. The tuning gain was **0.0296**, compared with the observed three-split AP spread of **0.0225**.
* I also considered the error analysis, including false positives, false negatives, and differences in performance across customer segments. These results help identify potential operational risks.
* Based on the measured performance and the predefined decision rule, I selected this model for the capstone. However, its performance should be validated on future campaign data before real-world deployment.

**Trade-offs**  
* Average Precision measures ranking quality, not the cost of every individual mistake.
* The 0.5 threshold is a diagnostic choice, not necessarily the best campaign threshold.
* The model does not prove that contacting a customer causes subscription.
* The feature `duration` was excluded because it is unavailable before the call.
* Results need validation on future campaign data before operational deployment.

# **Part F — Close the day**

**Closing Standup**  
**Done :-**
* Loaded the bank marketing dataset into a fresh notebook.
* Rebuilt the preprocessing and baseline pipelines.
* Tuned Random Forest with a precommitted search budget.
* Compared Cross-Validation and holdout Average Precision (AP).
* Measured score variation across three train-test splits.
* Inspected high-confidence errors and customer segments.
* Selected a provisional final model based on measured results.

**Blocked / Follow-up :-**
* Validate performance on future campaign data.
* Investigate whether a different outreach threshold improves campaign utility.
* Review error patterns and potential operational bias before deployment.

**Final Result :-**
* Selected Model - Tuned Random Forest
* Untuned Randome Forest AP - 0.4284
* Tuned Random Forest AP :- 0.4580
* Tuning Gain :- 0.0296
* Three-Split AP Spread :- 0.0225

**Next Step**  
* Document the final evidence and validate the chosen model before any real-world use.